# TP Pandas Enrichi — Corrigé (Dataset Iris)

Ce corrigé couvre toutes les parties du TP enrichi avec des explications.

## Partie 1 — Chargement et exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

# Chargement du dataset
iris = load_iris()

# Construction du DataFrame
df = pd.DataFrame(iris.data, columns=["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"])
df["Species"] = [iris.target_names[i] for i in iris.target]

print("=== 5 premières lignes ===")
display(df.head())

print("\n=== 5 dernières lignes ===")
display(df.tail())

In [ ]:
# Forme du DataFrame
print("Shape:", df.shape)
print(f"\n{df.shape[0]} lignes, {df.shape[1]} colonnes")

In [ ]:
# Info et describe
print("=== df.info() ===")
df.info()

print("\n=== df.describe() ===")
display(df.describe())

print("\n=== Types de données ===")
print(df.dtypes)

## Partie 2 — Sélections et indexation

In [ ]:
# 1. Sélection de colonnes
print("=== Colonnes SepalLengthCm et PetalLengthCm ===")
display(df[["SepalLengthCm", "PetalLengthCm"]].head())

In [ ]:
# 2. loc[] : sélection par labels
print("=== loc[10:20, ['Species', 'PetalWidthCm']] ===")
display(df.loc[10:20, ["Species", "PetalWidthCm"]])

In [ ]:
# 3. iloc[] : sélection par position
print("=== iloc[:5, :2] (5 premières lignes, 2 premières colonnes) ===")
display(df.iloc[:5, :2])

In [ ]:
# 4. Filtrage simple
print("=== Fleurs avec SepalLengthCm > 6.0 ===")
df_grand = df[df["SepalLengthCm"] > 6.0]
print(f"{len(df_grand)} fleurs trouvées")
display(df_grand.head())

In [ ]:
# 5. Filtrage avec plusieurs conditions
print("=== SepalLengthCm > 5.5 ET PetalLengthCm < 4.0 ===")
df_filtre = df[(df["SepalLengthCm"] > 5.5) & (df["PetalLengthCm"] < 4.0)]
print(f"{len(df_filtre)} fleurs trouvées")
display(df_filtre.head())

In [ ]:
# 6. Méthode query()
print("=== Espèce versicolor avec query() ===")
df_versicolor = df.query("Species == 'versicolor'")
print(f"{len(df_versicolor)} fleurs versicolor")
display(df_versicolor.head())

## Partie 3 — Manipulation des colonnes et lignes

In [ ]:
# Travail sur une copie pour ne pas modifier l'original
df_work = df.copy()

# 1. Ajout de PetalRatio
df_work["PetalRatio"] = df_work["PetalLengthCm"] / df_work["PetalWidthCm"]

# 2. Ajout de SepalArea
df_work["SepalArea"] = df_work["SepalLengthCm"] * df_work["SepalWidthCm"]

print("=== Nouvelles colonnes ajoutées ===")
display(df_work.head())

In [ ]:
# 3. Colonne catégorielle Taille
def categorize_size(length):
    if length < 5.0:
        return "petite"
    elif length < 6.5:
        return "moyenne"
    else:
        return "grande"

df_work["Taille"] = df_work["SepalLengthCm"].apply(categorize_size)

# Alternative avec np.select (plus efficace)
# conditions = [
#     df_work["SepalLengthCm"] < 5.0,
#     df_work["SepalLengthCm"] < 6.5
# ]
# choices = ["petite", "moyenne"]
# df_work["Taille"] = np.select(conditions, choices, default="grande")

print("=== Distribution des tailles ===")
print(df_work["Taille"].value_counts())

In [ ]:
# 4. Suppression de la colonne SepalArea
df_work = df_work.drop(columns=["SepalArea"])
print("Colonnes après suppression:", list(df_work.columns))

In [ ]:
# 5. Suppression des lignes où SepalLengthCm < 5.0
print(f"Avant suppression: {len(df_work)} lignes")
df_work = df_work[df_work["SepalLengthCm"] >= 5.0]
print(f"Après suppression: {len(df_work)} lignes")

In [ ]:
# 6. Tri par PetalLengthCm décroissant
df_work = df_work.sort_values(by="PetalLengthCm", ascending=False)
print("=== Trié par PetalLengthCm décroissant ===")
display(df_work.head())

In [ ]:
# 7. Réinitialisation de l'index
df_work = df_work.reset_index(drop=True)
print("=== Index réinitialisé ===")
display(df_work.head())

## Partie 4 — Gestion des valeurs manquantes

In [ ]:
# Travail sur une nouvelle copie
df_nan = df.copy()

# 1. Introduction de NaN artificiels
np.random.seed(42)  # Pour reproductibilité
indices_nan = df_nan.sample(5).index
df_nan.loc[indices_nan, "PetalWidthCm"] = np.nan

print(f"Indices avec NaN: {list(indices_nan)}")

In [ ]:
# 2. Compter les valeurs manquantes
print("=== Valeurs manquantes par colonne ===")
print(df_nan.isna().sum())

In [ ]:
# 3. Afficher les lignes avec NaN
print("=== Lignes contenant des NaN ===")
display(df_nan[df_nan.isna().any(axis=1)])

In [ ]:
# 4. Remplacement par la moyenne
mean_petal_width = df_nan["PetalWidthCm"].mean()
print(f"Moyenne de PetalWidthCm: {mean_petal_width:.3f}")

df_filled = df_nan.copy()
df_filled["PetalWidthCm"] = df_filled["PetalWidthCm"].fillna(mean_petal_width)

print("\nAprès fillna:")
print(df_filled.isna().sum())

In [ ]:
# 5. Suppression des lignes avec NaN (sur une copie)
df_dropped = df_nan.dropna()
print(f"Avant dropna: {len(df_nan)} lignes")
print(f"Après dropna: {len(df_dropped)} lignes")

## Partie 5 — Agrégations et groupby

In [ ]:
# 1. Comptage par espèce
print("=== Occurrences par espèce ===")
print(df["Species"].value_counts())

In [ ]:
# 2. Moyenne par espèce
print("=== Moyenne par espèce ===")
display(df.groupby("Species").mean(numeric_only=True))

In [ ]:
# 3. Plusieurs statistiques par espèce
print("=== Statistiques multiples par espèce ===")
stats = df.groupby("Species").agg(["mean", "std", "min", "max"])
display(stats)

In [ ]:
# 4. Percentiles par espèce
print("=== Percentiles de PetalLengthCm par espèce ===")
percentiles = df.groupby("Species")["PetalLengthCm"].quantile([0.25, 0.5, 0.75]).unstack()
percentiles.columns = ["Q1 (25%)", "Médiane (50%)", "Q3 (75%)"]
display(percentiles)

In [ ]:
# 5. Tableau croisé Species x Taille
# D'abord ajouter la colonne Taille
df_with_size = df.copy()
df_with_size["Taille"] = df_with_size["SepalLengthCm"].apply(
    lambda x: "petite" if x < 5.0 else ("moyenne" if x < 6.5 else "grande")
)

print("=== Tableau croisé Species × Taille ===")
cross_tab = pd.crosstab(df_with_size["Species"], df_with_size["Taille"])
display(cross_tab)

## Partie 6 — Lecture/écriture de fichiers

In [ ]:
# 1. Export en CSV
df.to_csv("iris_enrichi.csv", index=False)
print("Fichier iris_enrichi.csv créé !")

In [ ]:
# 2. Rechargement du CSV
df_reload = pd.read_csv("iris_enrichi.csv")
print("=== Fichier rechargé ===")
display(df_reload.head())

In [ ]:
# 3. Vérification d'égalité
print("DataFrames identiques ?", df.equals(df_reload))

# Comparaison plus détaillée si nécessaire
print(f"Mêmes shapes: {df.shape == df_reload.shape}")
print(f"Mêmes colonnes: {list(df.columns) == list(df_reload.columns)}")

In [ ]:
# 4. Export avec séparateur ;
df.to_csv("iris_semicolon.csv", index=False, sep=";")

# Rechargement avec le bon séparateur
df_semicolon = pd.read_csv("iris_semicolon.csv", sep=";")
print("=== Rechargement avec sep=';' ===")
display(df_semicolon.head())

## Partie 7 — Visualisations

In [ ]:
# 7.1 Histogramme
plt.figure(figsize=(8, 5))
plt.hist(df["SepalLengthCm"], bins=15, edgecolor="black", alpha=0.7)
plt.title("Distribution de SepalLengthCm")
plt.xlabel("SepalLengthCm")
plt.ylabel("Fréquence")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 7.2 Nuage de points simple
plt.figure(figsize=(8, 6))
plt.scatter(df["SepalLengthCm"], df["PetalLengthCm"], alpha=0.7)
plt.title("SepalLengthCm vs PetalLengthCm")
plt.xlabel("SepalLengthCm")
plt.ylabel("PetalLengthCm")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 7.3 Nuage de points coloré par espèce
colors = {"setosa": "#e74c3c", "versicolor": "#3498db", "virginica": "#2ecc71"}

plt.figure(figsize=(10, 7))
for species in df["Species"].unique():
    subset = df[df["Species"] == species]
    plt.scatter(
        subset["SepalLengthCm"], 
        subset["PetalLengthCm"], 
        c=colors[species], 
        label=species,
        alpha=0.7,
        s=50
    )

plt.title("SepalLengthCm vs PetalLengthCm (par espèce)")
plt.xlabel("SepalLengthCm")
plt.ylabel("PetalLengthCm")
plt.legend(title="Espèce")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 7.4 Boxplots par espèce
species_list = sorted(df["Species"].unique())
data_by_species = [df[df["Species"] == sp]["PetalLengthCm"].values for sp in species_list]

plt.figure(figsize=(8, 6))
bp = plt.boxplot(data_by_species, labels=species_list, patch_artist=True)

# Coloration des boîtes
box_colors = ["#e74c3c", "#3498db", "#2ecc71"]
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

plt.title("PetalLengthCm par espèce")
plt.xlabel("Espèce")
plt.ylabel("PetalLengthCm")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 7.5 Diagramme en barres
counts = df["Species"].value_counts()

plt.figure(figsize=(8, 5))
bars = plt.bar(counts.index, counts.values, color=["#e74c3c", "#3498db", "#2ecc71"], edgecolor="black")

# Ajout des valeurs sur les barres
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val), ha="center", fontweight="bold")

plt.title("Nombre de fleurs par espèce")
plt.xlabel("Espèce")
plt.ylabel("Effectif")
plt.ylim(0, 60)
plt.tight_layout()
plt.show()

In [ ]:
# 7.6 Matrice de corrélation
numeric_cols = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
im = plt.imshow(corr_matrix, cmap="RdYlBu_r", vmin=-1, vmax=1)
plt.colorbar(im, label="Corrélation")

# Labels
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)

# Valeurs dans les cellules
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        text = f"{corr_matrix.iloc[i, j]:.2f}"
        plt.text(j, i, text, ha="center", va="center", fontweight="bold",
                 color="white" if abs(corr_matrix.iloc[i, j]) > 0.5 else "black")

plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

In [ ]:
# 7.7 Subplots multiples (2x2 histogrammes)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, col in enumerate(numeric_cols):
    ax = axes[idx // 2, idx % 2]
    ax.hist(df[col], bins=15, edgecolor="black", alpha=0.7, color="#3498db")
    ax.set_title(f"Distribution de {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Fréquence")
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Histogrammes des 4 variables numériques", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Partie 8 — Exercices bonus

In [ ]:
# Bonus 1 : Outlier avec le plus grand PetalRatio
df_bonus = df.copy()
df_bonus["PetalRatio"] = df_bonus["PetalLengthCm"] / df_bonus["PetalWidthCm"]

idx_max = df_bonus["PetalRatio"].idxmax()
print("=== Fleur avec le plus grand PetalRatio ===")
display(df_bonus.loc[[idx_max]])

print(f"\nPetalRatio max: {df_bonus.loc[idx_max, 'PetalRatio']:.2f}")
print("C'est une fleur setosa, ce qui est cohérent car les setosa ont des pétales très fins.")

In [ ]:
# Bonus 2 : Coefficient de variation par espèce
def coef_variation(x):
    return x.std() / x.mean() * 100  # En pourcentage

cv_by_species = df.groupby("Species")[numeric_cols].apply(coef_variation)
print("=== Coefficient de variation (%) par espèce ===")
display(cv_by_species.round(2))

In [ ]:
# Bonus 3 : Fleurs "typiques"
# Une fleur est typique si toutes ses mesures sont dans mean ± 1 std de son espèce

def is_typical(row):
    species = row["Species"]
    species_data = df[df["Species"] == species][numeric_cols]
    means = species_data.mean()
    stds = species_data.std()
    
    for col in numeric_cols:
        if not (means[col] - stds[col] <= row[col] <= means[col] + stds[col]):
            return False
    return True

df_bonus["Typique"] = df_bonus.apply(is_typical, axis=1)

print("=== Fleurs typiques vs atypiques ===")
print(df_bonus["Typique"].value_counts())
print(f"\n{df_bonus['Typique'].mean()*100:.1f}% des fleurs sont typiques")

In [ ]:
# Nettoyage des fichiers temporaires
import os
for f in ["iris_enrichi.csv", "iris_semicolon.csv"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Fichier {f} supprimé.")